In [8]:
# Pre-specify 1:2 as the primary match; retain 1:1 and 1:3 for robustness.
# Match on event year and Bloomberg industry, then size with a sector preference.

In [9]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/bankruptcy_events.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = project_paths(ROOT).reports
PRIMARY_MATCH_RATIO = 2
ROBUSTNESS_MATCH_RATIOS = (1, 3)
MATCH_RATIOS = tuple(sorted({PRIMARY_MATCH_RATIO, *ROBUSTNESS_MATCH_RATIOS}))
MAX_CONTROL_REUSE = 3
MAX_MARKET_CAP_AGE_DAYS = 45
MAX_ASSETS_AGE_MONTHS = 18
REQUIRED_COLUMNS = [
    "bankrupt_company_id", "control_company_id", "bankrupt_prediction_date",
    "control_pseudo_event_date", "prediction_year", "bankrupt_sector",
    "control_sector", "bankrupt_industry", "control_industry", "bankrupt_size",
    "control_size", "size_distance", "match_stage", "sector_match",
    "industry_match", "year_match", "control_reuse_count",
]
AUDIT_COLUMNS = ["size_basis", "bankrupt_size_date", "control_size_date",
                 "control_prediction_year", "match_rank",
                 "bankrupt_log_market_cap", "control_log_market_cap",
                 "bankrupt_log_total_assets", "control_log_total_assets",
                 "bankrupt_financial_history_years", "control_financial_history_years",
                 "bankrupt_market_history_months", "control_market_history_months"]

bankruptcy_events = pd.read_csv(PROCESSED / "bankruptcy_events.csv",
                                parse_dates=["prediction_date", "outcome_end_date"])
control_events = pd.read_csv(PROCESSED / "nonbankruptcy_candidate_events.csv",
                             parse_dates=["prediction_date", "outcome_end_date"])
metadata = pd.read_csv(PROCESSED / "company_availability.csv")
if metadata["company_id"].duplicated().any():
    raise ValueError("Company metadata must be unique by company_id")
for column in ["bloomberg_sector", "bloomberg_industry", "bloomberg_identifier"]:
    metadata[column] = metadata[column].astype("string").str.strip()
    if metadata[column].isna().any() or metadata[column].eq("").any():
        raise ValueError(f"Missing {column}; correct cleaning before matching")
metadata["security_key"] = metadata["bloomberg_identifier"].str.upper().str.replace(r"\s+", " ", regex=True)
metadata["sector_key"] = metadata["bloomberg_sector"].str.casefold()
metadata["industry_key"] = metadata["bloomberg_industry"].str.casefold()
if metadata.loc[metadata["company_type"].eq("nonbankrupt"), "security_key"].duplicated().any():
    raise ValueError("Duplicate non-bankrupt securities need identity review before reuse can be capped")
for events, expected_target, expected_type in [
    (bankruptcy_events, 1, "bankrupt"), (control_events, 0, "nonbankrupt")
]:
    if events.empty:
        raise ValueError(f"No {expected_type} events; rerun notebook 05")
    if events[["company_id", "prediction_date", "outcome_end_date"]].isna().any().any():
        raise ValueError("Missing event identifiers or dates")
    if events.duplicated(["company_id", "prediction_date"]).any():
        raise ValueError("Duplicate company/prediction-date events")
    if not events["target"].eq(expected_target).all():
        raise ValueError("Unexpected event targets")
    if not events["prediction_year"].eq(events["prediction_date"].dt.year).all():
        raise ValueError("Prediction years do not match event dates")
    if not events["outcome_end_date"].eq(events["prediction_date"] + pd.DateOffset(months=12)).all():
        raise ValueError("Events must have 12-month outcome windows")
    types = events["company_id"].map(metadata.set_index("company_id")["company_type"])
    if not types.eq(expected_type).all():
        raise ValueError("Unknown event companies or inconsistent company types")


In [10]:
# Load only market cap and assets; retain zeros until snapshot eligibility.
size_chunks = []
for chunk in pd.read_csv(PROCESSED / "clean_company_long.csv",
                         usecols=["company_id", "date", "field", "value"], chunksize=250_000):
    selected = chunk.loc[chunk["field"].isin(["market_cap", "total_assets", "adjusted_price"])].copy()
    selected["value"] = pd.to_numeric(selected["value"], errors="coerce")
    size_chunks.append(selected.loc[selected["value"].notna() & np.isfinite(selected["value"])])
size_history = pd.concat(size_chunks, ignore_index=True)
size_history["date"] = pd.to_datetime(size_history["date"], errors="raise").dt.normalize()
if size_history.duplicated(["company_id", "date", "field"]).any():
    raise ValueError("Duplicate size observations; rerun cleaning")
size_histories = {}
for (company_id, field), group in size_history.groupby(["company_id", "field"], sort=False):
    group = group.sort_values("date")
    size_histories[(company_id, field)] = (
        group["date"].to_numpy(dtype="datetime64[ns]"), group["value"].to_numpy(dtype=float))

financial_year_starts = {}
financial_rows = size_history.loc[size_history["field"].eq("total_assets") & size_history["value"].ge(0)]
for company_id, group in financial_rows.groupby("company_id", sort=False):
    dates = np.sort(group["date"].drop_duplicates().to_numpy(dtype="datetime64[ns]"))
    years = dates.astype("datetime64[Y]")
    financial_year_starts[company_id] = dates[np.r_[True, years[1:] != years[:-1]]]
market_month_starts = {}
market_rows = size_history.loc[size_history["field"].eq("adjusted_price") & size_history["value"].gt(0)]
for company_id, group in market_rows.groupby("company_id", sort=False):
    dates = np.sort(group["date"].drop_duplicates().to_numpy(dtype="datetime64[ns]"))
    months = dates.astype("datetime64[M]")
    market_month_starts[company_id] = dates[np.r_[True, months[1:] != months[:-1]]]


def add_size_snapshots(events, company_metadata, histories):
    result = events.merge(company_metadata[["company_id", "bloomberg_sector", "bloomberg_industry",
                                            "security_key", "sector_key", "industry_key"]],
                          on="company_id", how="left", validate="many_to_one").reset_index(drop=True)
    result["financial_history_years"] = 0
    result["market_history_months"] = 0
    for company_id, indices in result.groupby("company_id", sort=False).indices.items():
        query = result.loc[indices, "prediction_date"].to_numpy(dtype="datetime64[ns]")
        result.loc[indices, "financial_history_years"] = np.searchsorted(
            financial_year_starts.get(company_id, np.array([], dtype="datetime64[ns]")), query, side="left")
        result.loc[indices, "market_history_months"] = np.searchsorted(
            market_month_starts.get(company_id, np.array([], dtype="datetime64[ns]")), query, side="left")
    for field in ["market_cap", "total_assets"]:
        values = np.full(len(result), np.nan)
        observed_dates = np.full(len(result), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")
        for company_id, indices in result.groupby("company_id", sort=False).indices.items():
            history_dates, history_values = histories.get(
                (company_id, field), (np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)))
            query = result.loc[indices, "prediction_date"].to_numpy(dtype="datetime64[ns]")
            positions = np.searchsorted(history_dates, query, side="left") - 1
            available = positions >= 0
            if available.any():
                values[indices[available]] = history_values[positions[available]]
                observed_dates[indices[available]] = history_dates[positions[available]]
        result[field + "_value"] = values
        result[field + "_date"] = observed_dates
        if field == "market_cap":
            earliest = result["prediction_date"] - pd.Timedelta(days=MAX_MARKET_CAP_AGE_DAYS)
        else:
            earliest = result["prediction_date"] - pd.DateOffset(months=MAX_ASSETS_AGE_MONTHS)
        usable = (result[field + "_value"].gt(0)
                  & result[field + "_date"].ge(earliest)
                  & result[field + "_date"].lt(result["prediction_date"]))
        result[field + "_usable"] = usable
        result[field + "_log"] = np.nan
        result.loc[usable, field + "_log"] = np.log(result.loc[usable, field + "_value"])
    result["has_size"] = result[["market_cap_usable", "total_assets_usable"]].any(axis=1)
    return result

bankrupt_snapshots = add_size_snapshots(bankruptcy_events, metadata, size_histories)
control_snapshots = add_size_snapshots(control_events, metadata, size_histories)


In [11]:
# Index candidate events by exact Bloomberg classification and year.
industry_groups = control_snapshots.groupby(["prediction_year", "industry_key"], sort=False).indices
sector_groups = control_snapshots.groupby(["prediction_year", "sector_key"], sort=False).indices


def ranked_control_pool(bank, controls, industry_index, sector_index):
    indices = []
    for year in [bank.prediction_year, bank.prediction_year - 1, bank.prediction_year + 1]:
        for groups, key in [(industry_index, bank.industry_key), (sector_index, bank.sector_key)]:
            found = groups.get((year, key))
            if found is not None:
                indices.append(found)
    if not indices:
        return pd.DataFrame(columns=REQUIRED_COLUMNS + AUDIT_COLUMNS)
    pool = controls.iloc[np.unique(np.concatenate(indices))].copy()
    pool = pool.loc[pool["security_key"].ne(bank.security_key) & pool["company_id"].ne(bank.company_id)]
    same_year = pool["prediction_year"].eq(bank.prediction_year)
    adjacent_year = pool["prediction_year"].sub(bank.prediction_year).abs().eq(1)
    same_industry = pool["industry_key"].eq(bank.industry_key)
    same_sector = pool["sector_key"].eq(bank.sector_key)
    stage = np.select([same_year & same_industry, same_year & same_sector,
                       adjacent_year & same_industry, adjacent_year & same_sector], [1, 2, 3, 4], default=0)
    use_market = bool(bank.market_cap_usable) & pool["market_cap_usable"]
    use_assets = ~use_market & bool(bank.total_assets_usable) & pool["total_assets_usable"]
    usable = (stage > 0) & (use_market | use_assets)
    pool["match_stage"] = stage
    pool["sector_match"] = same_sector.astype(bool)
    pool["industry_match"] = same_industry.astype(bool)
    pool["year_match"] = same_year.astype(bool)
    pool["control_reuse_count"] = 0
    pool["bankrupt_log_market_cap"] = bank.market_cap_log
    pool["control_log_market_cap"] = pool["market_cap_log"]
    pool["bankrupt_log_total_assets"] = bank.total_assets_log
    pool["control_log_total_assets"] = pool["total_assets_log"]
    pool["bankrupt_financial_history_years"] = bank.financial_history_years
    pool["control_financial_history_years"] = pool["financial_history_years"]
    pool["bankrupt_market_history_months"] = bank.market_history_months
    pool["control_market_history_months"] = pool["market_history_months"]
    pool["size_basis"] = np.where(use_market, "market_cap", "total_assets")
    pool["bankrupt_size"] = np.where(use_market, bank.market_cap_log, bank.total_assets_log)
    pool["control_size"] = np.where(use_market, pool["market_cap_log"], pool["total_assets_log"])
    pool["bankrupt_size_date"] = bank.total_assets_date
    pool.loc[use_market, "bankrupt_size_date"] = bank.market_cap_date
    pool["control_size_date"] = pool["market_cap_date"].where(use_market, pool["total_assets_date"])
    pool["size_distance"] = (pool["bankrupt_size"] - pool["control_size"]).abs()
    pool["sector_penalty"] = (~same_sector).astype(int)
    pool["year_distance"] = pool["prediction_year"].sub(bank.prediction_year).abs()
    pool["date_gap_days"] = pool["prediction_date"].sub(bank.prediction_date).dt.days.abs()
    pool = pool.loc[usable].copy()
    pool = pool.rename(columns={"company_id": "control_company_id", "prediction_date": "control_pseudo_event_date",
                                "prediction_year": "control_prediction_year", "bloomberg_sector": "control_sector",
                                "bloomberg_industry": "control_industry"})
    pool["bankrupt_company_id"] = bank.company_id
    pool["bankrupt_prediction_date"] = bank.prediction_date
    pool["prediction_year"] = bank.prediction_year
    pool["bankrupt_sector"] = bank.bloomberg_sector
    pool["bankrupt_industry"] = bank.bloomberg_industry
    # Keep each company's best eligible date for this bankrupt event. Reuse is
    # capped by company, so additional dates cannot evade the cap.
    return pool.sort_values(["match_stage", "size_distance", "sector_penalty", "year_distance", "date_gap_days",
                             "control_company_id", "control_pseudo_event_date"], kind="stable").drop_duplicates(
        "control_company_id").reset_index(drop=True)

ranked_pools = {}
ordering_rows = []
for index, bank in enumerate(bankrupt_snapshots.itertuples(index=False)):
    pool = ranked_control_pool(bank, control_snapshots, industry_groups, sector_groups)
    ranked_pools[index] = pool
    ordering_rows.append({"event_index": index, "bankrupt_company_id": bank.company_id,
                          "bankrupt_prediction_date": bank.prediction_date,
                          "stage1_companies": int(pool["match_stage"].eq(1).sum()),
                          "all_candidate_companies": len(pool)})
match_order = pd.DataFrame(ordering_rows).sort_values(
    ["stage1_companies", "all_candidate_companies", "bankrupt_prediction_date", "bankrupt_company_id"], kind="stable")


In [12]:
# Give every event a chance at its first control before allocating the next round.
# Pools already enforce the hierarchy and contain one best date per company.
from collections import Counter


def allocate_controls(pools, event_order, maximum_ratio, reuse_cap):
    usage = Counter()
    chosen = {index: set() for index in event_order}
    records = []
    for rank in range(1, maximum_ratio + 1):
        for index in event_order:
            pool = pools[index]
            for position, company_id in enumerate(pool["control_company_id"]):
                if company_id in chosen[index] or usage[company_id] >= reuse_cap:
                    continue
                record = pool.iloc[position][REQUIRED_COLUMNS + [column for column in AUDIT_COLUMNS if column != "match_rank"]].to_dict()
                record["match_rank"] = rank
                records.append(record)
                chosen[index].add(company_id)
                usage[company_id] += 1
                break
    return pd.DataFrame(records, columns=REQUIRED_COLUMNS + AUDIT_COLUMNS), usage

allocation, allocation_usage = allocate_controls(
    ranked_pools, match_order["event_index"].tolist(), max(MATCH_RATIOS), MAX_CONTROL_REUSE)

samples = {}
match_diagnostics = []
usage_parts = []
all_event_keys = pd.MultiIndex.from_frame(bankruptcy_events[["company_id", "prediction_date"]])
for ratio in MATCH_RATIOS:
    prefix = allocation.loc[allocation["match_rank"].le(ratio)].copy()
    counts = prefix.groupby(["bankrupt_company_id", "bankrupt_prediction_date"]).size()
    full_sets = counts.loc[counts.eq(ratio)].index
    prefix_keys = pd.MultiIndex.from_frame(prefix[["bankrupt_company_id", "bankrupt_prediction_date"]])
    sample = prefix.loc[prefix_keys.isin(full_sets)].sort_values(
        ["bankrupt_prediction_date", "bankrupt_company_id", "match_rank"], kind="stable").reset_index(drop=True)
    sample["control_reuse_count"] = sample.groupby("control_company_id")["control_company_id"].transform("size").astype(int)
    samples[ratio] = sample[REQUIRED_COLUMNS + AUDIT_COLUMNS]
    matched_event_counts = counts.reindex(all_event_keys, fill_value=0)
    diagnostics = {"ratio": f"1:{ratio}",
                   "sample_role": "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness",
                   "bankrupt_events": len(full_sets), "matched_pairs": len(sample),
                   "control_companies": sample["control_company_id"].nunique(),
                   "incomplete_bankrupt_events": int(matched_event_counts.lt(ratio).sum()),
                   "zero_control_events": int(matched_event_counts.eq(0).sum())}
    for stage in range(1, 5):
        diagnostics[f"stage_{stage}_pairs"] = int(sample["match_stage"].eq(stage).sum())
    match_diagnostics.append(diagnostics)
    usage = sample.groupby("control_company_id").size().rename("times_used").reset_index()
    usage["sample"] = f"1:{ratio}"
    usage_parts.append(usage)
match_summary = pd.DataFrame(match_diagnostics)
control_usage = pd.concat(usage_parts, ignore_index=True)
control_usage_report = control_usage.pivot(index="control_company_id", columns="sample", values="times_used").fillna(0).astype(int)
control_usage_report = control_usage_report.reindex(columns=[f"1:{ratio}" for ratio in MATCH_RATIOS], fill_value=0)
control_usage_report = control_usage_report.sort_values(
    [f"1:{ratio}" for ratio in reversed(MATCH_RATIOS)], ascending=False, kind="stable")
allocation_counts = allocation.groupby(["bankrupt_company_id", "bankrupt_prediction_date"]).size()
unmatched_events = bankruptcy_events[["company_id", "prediction_date"]].copy()
unmatched_events["allocated_controls"] = allocation_counts.reindex(all_event_keys, fill_value=0).to_numpy()


def standardized_mean_difference(bank_values, control_values):
    bank_values = np.asarray(bank_values, dtype=float)
    control_values = np.asarray(control_values, dtype=float)
    valid = np.isfinite(bank_values) & np.isfinite(control_values)
    bank_values, control_values = bank_values[valid], control_values[valid]
    if len(bank_values) < 2:
        return np.nan, np.nan, np.nan, len(bank_values)
    bank_mean, control_mean = bank_values.mean(), control_values.mean()
    pooled_sd = np.sqrt((bank_values.var(ddof=1) + control_values.var(ddof=1)) / 2)
    if pooled_sd == 0:
        smd = 0.0 if bank_mean == control_mean else np.inf
    else:
        smd = (bank_mean - control_mean) / pooled_sd
    return bank_mean, control_mean, smd, len(bank_values)


balance_rows = []
for ratio, sample in samples.items():
    sample_role = "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness"
    numeric_pairs = {
        "log_market_cap": ("bankrupt_log_market_cap", "control_log_market_cap"),
        "log_total_assets": ("bankrupt_log_total_assets", "control_log_total_assets"),
        "prediction_year": ("prediction_year", "control_prediction_year"),
        "financial_history_years": ("bankrupt_financial_history_years", "control_financial_history_years"),
        "market_history_months": ("bankrupt_market_history_months", "control_market_history_months"),
    }
    for variable, (bank_column, control_column) in numeric_pairs.items():
        bank_mean, control_mean, smd, n_pairs = standardized_mean_difference(
            sample[bank_column], sample[control_column])
        balance_rows.append({"ratio": f"1:{ratio}", "sample_role": sample_role,
            "variable": variable, "category": "", "bankrupt_mean": bank_mean,
            "control_mean": control_mean, "smd": smd, "abs_smd": abs(smd), "n_pairs": n_pairs})
    for variable, bank_column, control_column in [
        ("industry", "bankrupt_industry", "control_industry"),
        ("sector", "bankrupt_sector", "control_sector"),
    ]:
        bank_categories = sample[bank_column].astype("string").str.casefold()
        control_categories = sample[control_column].astype("string").str.casefold()
        valid = bank_categories.notna() & control_categories.notna()
        levels = sorted(set(bank_categories.loc[valid]) | set(control_categories.loc[valid]))
        for level in levels:
            bank_indicator = bank_categories.loc[valid].eq(level).astype(float)
            control_indicator = control_categories.loc[valid].eq(level).astype(float)
            bank_mean, control_mean, smd, n_pairs = standardized_mean_difference(bank_indicator, control_indicator)
            balance_rows.append({"ratio": f"1:{ratio}", "sample_role": sample_role,
                "variable": variable, "category": level, "bankrupt_mean": bank_mean,
                "control_mean": control_mean, "smd": smd, "abs_smd": abs(smd), "n_pairs": n_pairs})
matching_balance = pd.DataFrame(balance_rows)
matching_balance_summary = matching_balance.groupby(["ratio", "sample_role", "variable"], as_index=False).agg(
    max_abs_smd=("abs_smd", "max"), balance_terms=("abs_smd", "size"), matched_pairs=("n_pairs", "max"))


In [13]:
# Summarize ratio, match hierarchy, and reuse checks for each exported design.
check_rows = []
for ratio, sample in samples.items():
    counts = sample.groupby(["bankrupt_company_id", "bankrupt_prediction_date"]).size()
    same_year = sample["prediction_year"].eq(sample["control_prediction_year"])
    adjacent_year = sample["prediction_year"].sub(sample["control_prediction_year"]).abs().eq(1)
    same_industry = sample["industry_match"].astype(bool)
    same_sector = sample["sector_match"].astype(bool)
    expected_stage = np.select([same_year & same_industry, same_year & same_sector,
                                adjacent_year & same_industry, adjacent_year & same_sector], [1, 2, 3, 4], default=0)
    event_count_ok = len(counts) > 0 and counts.eq(ratio).all()
    unique_controls_ok = not sample.duplicated(
        ["bankrupt_company_id", "bankrupt_prediction_date", "control_company_id"]).any()
    stage_ok = sample["match_stage"].eq(expected_stage).all()
    reuse_ok = sample["control_reuse_count"].le(MAX_CONTROL_REUSE).all()
    check_rows.extend([
        {"ratio": f"1:{ratio}", "sample_role": "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness",
         "check": "complete_control_sets", "passed": bool(event_count_ok)},
        {"ratio": f"1:{ratio}", "sample_role": "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness",
         "check": "unique_controls_per_event", "passed": bool(unique_controls_ok)},
        {"ratio": f"1:{ratio}", "sample_role": "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness",
         "check": "match_stage_consistency", "passed": bool(stage_ok)},
        {"ratio": f"1:{ratio}", "sample_role": "primary" if ratio == PRIMARY_MATCH_RATIO else "robustness",
         "check": "maximum_control_reuse", "passed": bool(reuse_ok)},
    ])
matching_design_checks = pd.DataFrame(check_rows)


In [14]:
# Persist the samples, balance diagnostics, design checks, and reuse counts.
for ratio, sample in samples.items():
    path = PROCESSED / f"matched_event_sample_1to{ratio}.csv"
    sample.to_csv(path, index=False, date_format="%Y-%m-%d")
REPORTS.mkdir(parents=True, exist_ok=True)
matching_balance.to_csv(REPORTS / "matching_balance.csv", index=False)
matching_balance_summary.to_csv(REPORTS / "matching_balance_summary.csv", index=False)
match_summary.to_csv(REPORTS / "matching_design_summary.csv", index=False)
control_usage_report.to_csv(REPORTS / "control_reuse_summary.csv")
matching_design_checks.to_csv(REPORTS / "matching_design_checks.csv", index=False)
